# 09 Differential Expression (Run Once)

## Overview
This notebook runs differential expression to quantify infection-associated transcriptional changes within each annotated mouse cell type using the canonical all-cells object.

## DGE Contrast Clarification
- Source sample labels upstream: `OG` vs `Mock`
- Analysis labels in this notebook: `Infected` vs `Naive`
- Normalization used before DGE: `OG -> Infected`, `Mock -> Naive`
- Interpretation: positive `logFC` means higher in `Infected`; negative `logFC` means higher in `Naive`

## Statistical Method
- Per-cell-type test: `scanpy.tl.rank_genes_groups` with `method="wilcoxon"`
- Gene-level raw p-values exported as `p_val`
- Benjamini-Hochberg FDR correction exported as `p_adj`
- Underpowered strata are skipped when either group has fewer than two cells

## Purpose
- Produce stable DGE outputs reused by downstream interpretation and enrichment workflows
- Standardize gene-level statistics across `class`, `subclass`, `supertype`, and `cluster`

## Upstream Dependency
- `adatas/brain_allcells_allgenes.h5ad`
- Required metadata: `group`, `class`, `subclass`, `supertype`, `cluster`
- Metadata contract inherited from notebook `07_Analysis_Preflight.ipynb`

## Primary Outputs
- `Results/DGE/DGE_level_class_nb09.xlsx`
- `Results/DGE/DGE_level_subclass_nb09.xlsx`
- `Results/DGE/DGE_level_supertype_nb09.xlsx`
- `Results/DGE/DGE_level_cluster_nb09.xlsx`

## Output Structure
- Each workbook corresponds to one annotation level
- Each sheet corresponds to one eligible cell type at that level
- Each row is a gene with effect size (`logFC`) and significance metrics (`p_val`, `p_adj`)

## Scope Boundary
- Cell composition and count testing ownership: notebook `08`
- DGE visualization ownership: notebook `10`
- GSEA from saved DGE ownership: notebook `11`
- GSEA visualization ownership: notebook `12`
- Microglia branch follow-up: notebook `13`

## Standards
- Treat this as a run-once artifact notebook
- Re-run only when upstream data, annotations, or group assignments change
- Prefer using saved workbook outputs downstream instead of recomputing DGE unnecessarily

# Table of Contents

1. [Overview](#overview)
2. [DGE Contrast Clarification](#dge-contrast-clarification)
3. [Statistical Method](#statistical-method)
4. [Purpose](#purpose)
5. [Upstream Dependency](#upstream-dependency)
6. [Primary Outputs](#primary-outputs)
7. [Output Structure](#output-structure)
8. [Scope Boundary](#scope-boundary)
9. [Standards](#standards)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Final, TypedDict

import anndata as ad
import pandas as pd
import scanpy as sc
from statsmodels.stats.multitest import multipletests


@dataclass(frozen=True)
class Notebook09Config:
    """Centralized filesystem configuration for differential expression outputs."""

    analysis_root: Path
    dge_dir: Path


ANALYSIS_ROOT: Final[Path] = Path("/media/drive_c/Project_Brain_snRNAseq/Analysis")
REQUIRED_OBS: Final[list[str]] = ["group", "class", "subclass", "supertype", "cluster"]
GROUP_ALIAS_MAP: Final[dict[str, str]] = {"OG": "Infected", "Mock": "Naive"}

CONFIG = Notebook09Config(
    analysis_root=ANALYSIS_ROOT,
    dge_dir=ANALYSIS_ROOT / "Results" / "DGE",
)
CONFIG.dge_dir.mkdir(parents=True, exist_ok=True)


def resolve_input_path() -> Path:
    """Return the first existing canonical AnnData input path for notebook 09."""

    candidates = [
        CONFIG.analysis_root / "adatas" / "brain_allcells_allgenes.h5ad",
        CONFIG.analysis_root / "combined_filtered_pruning.h5ad",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    formatted = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Missing canonical AnnData input for notebook 09. Checked:\n" + formatted +
        "\nRun notebooks 01 and 02 before notebook 09."
    )


def resolve_mapping_path() -> Path:
    """Return a mapping CSV path used to recover missing supertype annotations."""

    candidates = [
        CONFIG.analysis_root / "Mapping" / "mapping_output_nb01.csv",
        CONFIG.analysis_root / "concat" / "Mapping" / "mapping_output_nb01.csv",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    formatted = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Missing mapping_output_nb01.csv for notebook 09. Checked:\n" + formatted +
        "\nRun notebook 01 or restore the mapping export before DGE."
    )


def maybe_attach_supertype_from_mapping(adata: ad.AnnData, mapping_csv: Path) -> None:
    """Backfill supertype metadata from mapping_output_nb01.csv when needed."""

    if "supertype" in adata.obs.columns and adata.obs["supertype"].notna().all():
        return
    if "supertype_name" in adata.obs.columns:
        adata.obs["supertype"] = adata.obs["supertype_name"].astype(str)
        return

    mapping_df = pd.read_csv(mapping_csv, comment="#", usecols=["cell_id", "supertype_name"])
    mapping_df = mapping_df.dropna(subset=["cell_id", "supertype_name"]).drop_duplicates("cell_id")
    mapping_series = mapping_df.set_index("cell_id")["supertype_name"].astype(str)
    aligned = mapping_series.reindex(adata.obs_names)

    if "supertype" not in adata.obs.columns:
        adata.obs["supertype"] = aligned.astype("object")
    else:
        missing_mask = adata.obs["supertype"].isna() | (adata.obs["supertype"].astype(str).str.len() == 0)
        adata.obs.loc[missing_mask, "supertype"] = aligned.loc[missing_mask]


def normalize_obs_columns(adata: ad.AnnData, mapping_csv: Path) -> None:
    """Normalize observation metadata columns to the canonical schema in-place."""

    rename_candidates: dict[str, list[str]] = {
        "sample": ["sample", "Sample"],
        "group": ["group", "treatment", "Study_Designation", "Condition"],
        "class": ["class", "class_name", "class_broad"],
        "subclass": ["subclass", "subclass_name"],
        "supertype": ["supertype", "supertype_name"],
        "cluster": ["cluster", "cluster_name", "cluster_alias"],
    }
    for target, options in rename_candidates.items():
        if target in adata.obs.columns:
            continue
        source: str | None = next((column for column in options if column in adata.obs.columns), None)
        if source is not None:
            adata.obs[target] = adata.obs[source].astype(str)

    maybe_attach_supertype_from_mapping(adata, mapping_csv)

    missing = [column for column in REQUIRED_OBS if column not in adata.obs.columns]
    if missing:
        raise KeyError(f"Missing required metadata columns after normalization: {missing}")

    adata.obs["group"] = adata.obs["group"].astype(str).replace(GROUP_ALIAS_MAP)


adata_path = resolve_input_path()
mapping_csv_path = resolve_mapping_path()
adata: ad.AnnData = sc.read_h5ad(str(adata_path))
normalize_obs_columns(adata, mapping_csv_path)

print(f"Using AnnData: {adata_path}")
print(f"Using mapping CSV: {mapping_csv_path}")
print(f"Loaded {adata.n_obs} cells x {adata.n_vars} genes")
adata

In [ ]:
class DgeRunResult(TypedDict):
    """Typed return container for one annotation-level DGE run."""

    dge_results: dict[str, pd.DataFrame]
    dge_summary: pd.DataFrame


def choose_group_labels(obs: pd.DataFrame, group_key: str = "group") -> tuple[str, str]:
    """Infer the test and control labels from the observation metadata."""

    groups = [str(value) for value in obs[group_key].dropna().astype(str).unique()]
    preferred_pairs = [
        ("Infected", "Naive"),
        ("OG", "Mock"),
        ("AD", "Ctrl"),
    ]
    for test_label, control_label in preferred_pairs:
        if test_label in groups and control_label in groups:
            return test_label, control_label
    if len(groups) == 2:
        groups_sorted = sorted(groups)
        return groups_sorted[1], groups_sorted[0]
    raise ValueError(f"Could not infer test/control groups from {groups}")


def sanitize_sheet_name(cell_type: str) -> str:
    """Return an Excel-safe sheet name for a cell-type label."""
    return str(cell_type).replace("/", "_")[:31]


def write_dge_workbook(output_path: Path, dge_results: dict[str, pd.DataFrame]) -> None:
    """Write one annotation-level DGE workbook with one sheet per cell type."""
    with pd.ExcelWriter(str(output_path), engine="openpyxl") as writer:
        for cell_type, dge_df in dge_results.items():
            dge_df.to_excel(writer, sheet_name=sanitize_sheet_name(cell_type), index=False)


def run_dge_per_annotation_level(
    adata: ad.AnnData,
    annotation_level: str,
    group_key: str = "group",
    test_label: str | None = None,
    control_label: str | None = None,
    output_dir: Path | None = None,
) -> DgeRunResult:
    """Run per-cell-type DGE for one annotation level and optionally export a workbook."""
    if annotation_level not in adata.obs.columns:
        raise KeyError(f"Missing annotation column: {annotation_level}")

    if test_label is None or control_label is None:
        test_label, control_label = choose_group_labels(adata.obs, group_key=group_key)
    print(f"Using comparison: {test_label} vs {control_label}")

    dge_results: dict[str, pd.DataFrame] = {}
    dge_summary_frames: list[pd.DataFrame] = []
    layer_used = "log_norm" if "log_norm" in adata.layers else None

    labels = adata.obs[annotation_level].astype(str)
    for cell_type in sorted(labels.dropna().unique()):
        adata_sub = adata[labels == cell_type].copy()
        groups = adata_sub.obs[group_key].dropna().astype(str).unique()
        if test_label not in groups or control_label not in groups:
            continue

        group_sizes = adata_sub.obs[group_key].astype(str).value_counts()
        n_test = int(group_sizes.get(test_label, 0))
        n_control = int(group_sizes.get(control_label, 0))
        if n_test < 2 or n_control < 2:
            print(
                f"Skipping {annotation_level}={cell_type}: insufficient cells for {test_label} vs {control_label} "
                f"(n_test={n_test}, n_control={n_control})"
            )
            continue

        try:
            sc.tl.rank_genes_groups(
                adata_sub,
                groupby=group_key,
                groups=[test_label],
                reference=control_label,
                method="wilcoxon",
                use_raw=False,
                layer=layer_used,
                n_genes=adata_sub.shape[1],
            )
        except ValueError as exc:
            print(f"Skipping {annotation_level}={cell_type}: {exc}")
            continue

        dge_df = sc.get.rank_genes_groups_df(adata_sub, group=test_label)
        dge_df = dge_df.rename(
            columns={
                "names": "gene",
                "logfoldchanges": "logFC",
                "pvals": "p_val",
                "pvals_adj": "p_adj_scanpy",
            }
        )
        dge_df["p_val"] = dge_df["p_val"].fillna(1.0)
        dge_df["p_adj"] = multipletests(dge_df["p_val"].to_numpy(), method="fdr_bh")[1]
        dge_df["cell_type"] = cell_type
        dge_df["test_group"] = test_label
        dge_df["control_group"] = control_label

        dge_results[cell_type] = dge_df
        dge_summary_frames.append(dge_df)

    dge_summary = pd.concat(dge_summary_frames, ignore_index=True) if dge_summary_frames else pd.DataFrame()

    if output_dir is not None:
        output_path = Path(output_dir) / f"DGE_level_{annotation_level}_nb09.xlsx"
        if dge_results:
            write_dge_workbook(output_path, dge_results)
            print(f"Saved: {output_path} (sheets={len(dge_results)})")
        else:
            print(
                f"Skipped writing {output_path}; no eligible cell types for {test_label} vs {control_label}"
            )

    return {"dge_results": dge_results, "dge_summary": dge_summary}


levels_to_run: list[str] = ["class", "subclass", "supertype", "cluster"]
dge_outputs: dict[str, DgeRunResult] = {}
for level in levels_to_run:
    print(f"Running DGE for {level}")
    dge_outputs[level] = run_dge_per_annotation_level(
        adata,
        annotation_level=level,
        output_dir=CONFIG.dge_dir,
    )